## Phase 1: Environment Setup & Data Pipeline
> This phase handles all imports, seeds, hyperparameters, and the data loading pipeline. CIFAR-10 is loaded directly from torchvision which caches it automatically on Kaggle — no manual download needed. Images are normalized to [-1, 1] because the diffusion model's noise schedule operates in that range and the final denoising output must match it.
The get_dataloader function applies standard augmentations: random horizontal flips add data diversity, and normalization with mean=0.5, std=0.5 maps the [0, 1] pixel range cleanly to [-1, 1].

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import math
import os
from tqdm import tqdm
from torchvision.utils import make_grid

torch.manual_seed(42)
np.random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 32
CHANNELS = 3
BATCH_SIZE = 128
LEARNING_RATE = 2e-4
TOTAL_TIMESTEPS = 1000
BETA_START = 1e-4
BETA_END = 0.02
EPOCHS = 100
BASE_CHANNELS = 64
DDIM_STEPS = 50
ETA = 0.0

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")


def get_dataloader(batch_size):
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    dataset = torchvision.datasets.CIFAR10(
        root="/kaggle/working/data",
        train=True,
        download=True,
        transform=transform
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )
    return loader


loader = get_dataloader(BATCH_SIZE)
print(f"Batches per epoch: {len(loader)}")
print(f"Total training samples: {len(loader.dataset)}")

Device: cuda
PyTorch: 2.10.0+cu128


100%|██████████| 170M/170M [00:05<00:00, 33.7MB/s] 


Batches per epoch: 390
Total training samples: 50000


## Phase 2: U-Net Architecture
>The U-Net is the neural backbone that learns to predict noise given a noisy image and a timestep. The architecture has four key components:
Time Embedding: Timestep t is encoded using sinusoidal positional embeddings (borrowed from the Transformer literature). These embeddings are then passed through two linear layers with SiLU activation, producing a dense time vector that gets injected into every residual block via scale-shift normalization.
Residual Blocks: Each residual block contains two Conv→GroupNorm→SiLU sequences. The time embedding is projected and added between them. GroupNorm with 8 groups is used instead of BatchNorm because it is stable at the small batch sizes that appear during inference (a single sample at generation time).
Attention Blocks: Self-attention is applied at the 16×16 bottleneck spatial resolution. This is where the model builds global coherence — learning that a "car" needs consistent color, shape, and context across the entire image. Applying attention at full 32×32 resolution would quadratically explode memory.
Encoder/Decoder with Skip Connections: The encoder downsamples via strided convolutions (preserving more information than max-pooling), the bottleneck applies two residual blocks with attention, and the decoder upsamples via bilinear interpolation followed by convolution. Skip connections concatenate encoder features to decoder features, which is why U-Nets excel at pixel-level reconstruction tasks.


In [2]:
def sinusoidal_embedding(timesteps, dim):
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, dtype=torch.float32) / (half - 1)
    ).to(timesteps.device)
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim * 4)
        )

    def forward(self, t):
        emb = sinusoidal_embedding(t, self.dim)
        return self.proj(emb)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim, groups=8):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(groups, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_channels)
        self.time_proj = nn.Linear(time_dim, out_channels * 2)
        self.act = nn.SiLU()
        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        t_out = self.time_proj(t_emb)[:, :, None, None]
        scale, shift = t_out.chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.act(h)
        h = self.conv2(h)
        return h + self.skip(x)


class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)
        head_dim = C // self.num_heads

        def reshape(t):
            t = t.view(B, self.num_heads, head_dim, H * W)
            return t.permute(0, 1, 3, 2)

        q, k, v = reshape(q), reshape(k), reshape(v)
        scale = head_dim ** -0.5
        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * scale, dim=-1)
        out = torch.matmul(attn, v)
        out = out.permute(0, 1, 3, 2).contiguous().view(B, C, H, W)
        return x + self.proj(out)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, use_attn=False):
        super().__init__()
        self.res1 = ResidualBlock(in_ch, out_ch, time_dim)
        self.res2 = ResidualBlock(out_ch, out_ch, time_dim)
        self.attn = AttentionBlock(out_ch) if use_attn else nn.Identity()
        self.downsample = nn.Conv2d(out_ch, out_ch, 4, 2, 1)

    def forward(self, x, t):
        x = self.res1(x, t)
        x = self.res2(x, t)
        x = self.attn(x)
        return self.downsample(x), x


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, time_dim, use_attn=False):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.res1 = ResidualBlock(in_ch + skip_ch, out_ch, time_dim)
        self.res2 = ResidualBlock(out_ch, out_ch, time_dim)
        self.attn = AttentionBlock(out_ch) if use_attn else nn.Identity()

    def forward(self, x, skip, t):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        x = self.res1(x, t)
        x = self.res2(x, t)
        return self.attn(x)


class UNet(nn.Module):
    def __init__(self, img_channels=3, base_ch=64, time_dim=256):
        super().__init__()
        ch = base_ch
        self.time_embed = TimeEmbedding(time_dim)

        self.init_conv = nn.Conv2d(img_channels, ch, 3, padding=1)

        self.down1 = DownBlock(ch, ch, time_dim * 4, use_attn=False)
        self.down2 = DownBlock(ch, ch * 2, time_dim * 4, use_attn=False)
        self.down3 = DownBlock(ch * 2, ch * 4, time_dim * 4, use_attn=True)

        self.bot1 = ResidualBlock(ch * 4, ch * 4, time_dim * 4)
        self.bot_attn = AttentionBlock(ch * 4)
        self.bot2 = ResidualBlock(ch * 4, ch * 4, time_dim * 4)

        self.up3 = UpBlock(ch * 4, ch * 4, ch * 2, time_dim * 4, use_attn=True)
        self.up2 = UpBlock(ch * 2, ch * 2, ch, time_dim * 4, use_attn=False)
        self.up1 = UpBlock(ch, ch, ch, time_dim * 4, use_attn=False)

        self.out_norm = nn.GroupNorm(8, ch)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(ch, img_channels, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)

        x = self.init_conv(x)

        x, s1 = self.down1(x, t_emb)
        x, s2 = self.down2(x, t_emb)
        x, s3 = self.down3(x, t_emb)

        x = self.bot1(x, t_emb)
        x = self.bot_attn(x)
        x = self.bot2(x, t_emb)

        x = self.up3(x, s3, t_emb)
        x = self.up2(x, s2, t_emb)
        x = self.up1(x, s1, t_emb)

        return self.out_conv(self.out_act(self.out_norm(x)))


model = UNet(img_channels=CHANNELS, base_ch=BASE_CHANNELS).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params / 1e6:.2f}M")

Trainable parameters: 13.93M


## Phase 3: Diffusion Process & DDIM Sampler
>This is the mathematical core of the entire system. It is split into two logical sub-components: the forward process (adding noise during training) and the reverse process (DDIM sampling during inference).
The Forward Process (Training)
The noise schedule defines β_t for t = 1…T, linearly increasing from β_start=1e-4 to β_end=0.02. From these, we precompute α_t = 1 - β_t and ᾱ_t = ∏α_s (the cumulative product). The key identity that makes diffusion training efficient is:

q(xₜ | x₀) = N(xₜ ; √ᾱₜ · x₀ , (1 − ᾱₜ)·I)

This means we can jump from a clean image x₀ to any arbitrary noisy step xₜ in a single operation, without iterating through all previous steps. The q_sample function does exactly this: it samples Gaussian noise ε and returns √ᾱₜ · x₀ + √(1−ᾱₜ) · ε. The U-Net is then trained to predict ε from xₜ and t, minimizing simple MSE loss.
The Reverse Process — Why DDIM Solves the Bottleneck
Standard DDPM reversal requires iterating from t=T all the way down to t=0, one step at a time. With T=1000, generating a single image means 1000 forward passes through the U-Net — roughly 30–60 seconds per image on a T4. This is the core inference bottleneck.
DDIM (Song et al., 2020) reformulates the reverse process as a non-Markovian generative process. The key insight is that DDIM defines a family of reverse processes that all share the same training objective as DDPM. This means you train identically, but at inference time you can skip arbitrary steps.
The DDIM reverse update from step t to step t_prev is:

x_{t−1} = √ᾱ_{t−1} · ( xₜ − √(1−ᾱₜ)·ε_θ ) / √ᾱₜ  +  √(1−ᾱ_{t−1} − σₜ²)·ε_θ + σₜ·z

When η=0 (which we use), σₜ=0 and the process becomes fully deterministic — the same noise vector always produces the same image. This determinism is also useful for interpolation and editing. With 50 steps instead of 1000, inference is 20× faster.


In [3]:
class DiffusionModel:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.T = timesteps
        self.device = device

        betas = torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

        self.register(betas, "betas")
        self.register(alphas_cumprod, "alphas_cumprod")
        self.register(alphas_cumprod_prev, "alphas_cumprod_prev")
        self.register(torch.sqrt(alphas_cumprod), "sqrt_alphas_cumprod")
        self.register(torch.sqrt(1.0 - alphas_cumprod), "sqrt_one_minus_alphas_cumprod")

    def register(self, tensor, name):
        setattr(self, name, tensor.to(self.device))

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_a = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_1ma = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_a * x0 + sqrt_1ma * noise, noise

    def p_loss(self, model, x0, t):
        noise = torch.randn_like(x0)
        xt, noise_target = self.q_sample(x0, t, noise)
        pred_noise = model(xt, t)
        return F.mse_loss(pred_noise, noise_target)

    @torch.no_grad()
    def ddim_sample(self, model, n_samples, img_size, channels, ddim_steps=50, eta=0.0):
        model.eval()
        step_size = self.T // ddim_steps
        timesteps = list(reversed(range(0, self.T, step_size)))

        x = torch.randn(n_samples, channels, img_size, img_size, device=self.device)

        for i, t_val in enumerate(tqdm(timesteps, desc="DDIM Sampling")):
            t_batch = torch.full((n_samples,), t_val, device=self.device, dtype=torch.long)

            pred_noise = model(x, t_batch)

            ac_t = self.alphas_cumprod[t_val]
            ac_prev = self.alphas_cumprod[timesteps[i + 1]] if i + 1 < len(timesteps) else torch.tensor(1.0, device=self.device)

            x0_pred = (x - torch.sqrt(1 - ac_t) * pred_noise) / torch.sqrt(ac_t)
            x0_pred = torch.clamp(x0_pred, -1, 1)

            sigma = eta * torch.sqrt((1 - ac_prev) / (1 - ac_t) * (1 - ac_t / ac_prev))
            dir_xt = torch.sqrt(1 - ac_prev - sigma ** 2) * pred_noise
            noise = torch.randn_like(x) if eta > 0 else torch.zeros_like(x)

            x = torch.sqrt(ac_prev) * x0_pred + dir_xt + sigma * noise

        model.train()
        return x


diffusion = DiffusionModel(
    timesteps=TOTAL_TIMESTEPS,
    beta_start=BETA_START,
    beta_end=BETA_END,
    device=DEVICE
)
print("Diffusion model initialized.")
print(f"Alpha_cumprod at t=0: {diffusion.alphas_cumprod[0].item():.4f}")
print(f"Alpha_cumprod at t=999: {diffusion.alphas_cumprod[999].item():.6f}")

Diffusion model initialized.
Alpha_cumprod at t=0: 0.9999
Alpha_cumprod at t=999: 0.000040


## Phase 4: Training Loop
>The training loop is a standard PyTorch loop with one diffusion-specific detail: for each batch of real images x0, we sample a random timestep t uniformly from {1, …, T} for each image independently. This is critical — the model must learn to denoise at all noise levels simultaneously, not just one.
We use the AdamW optimizer with lr=2e-4 and weight_decay=1e-4. A cosine annealing learning rate schedule prevents the loss from plateauing by smoothly decaying the learning rate over the training run. Gradient clipping at norm 1.0 prevents occasional exploding gradient issues that can occur in the early stages of training when the model has no sense of the noise distribution yet.
The checkpoint saving logic keeps both the best-loss model and the latest model — on Kaggle, sessions can be interrupted, so the latest_checkpoint.pt lets you resume without losing progress.

In [4]:
def save_checkpoint(model, optimizer, epoch, loss, path):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "loss": loss
    }, path)


def load_checkpoint(model, optimizer, path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt["epoch"], ckpt["loss"]


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/samples", exist_ok=True)

train_losses = []
best_loss = float("inf")

print("Starting training...")
print(f"Epochs: {EPOCHS} | Batch size: {BATCH_SIZE} | Device: {DEVICE}")


for epoch in range(EPOCHS):
    model.train()
    epoch_losses = []

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch_idx, (images, _) in enumerate(pbar):
        images = images.to(DEVICE)
        t = torch.randint(0, TOTAL_TIMESTEPS, (images.shape[0],), device=DEVICE).long()

        loss = diffusion.p_loss(model, images, t)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    scheduler.step()
    avg_loss = np.mean(epoch_losses)
    train_losses.append(avg_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    save_checkpoint(model, optimizer, epoch, avg_loss,
                    "/kaggle/working/checkpoints/latest_checkpoint.pt")

    if avg_loss < best_loss:
        best_loss = avg_loss
        save_checkpoint(model, optimizer, epoch, avg_loss,
                        "/kaggle/working/checkpoints/best_model.pt")
        print(f"New best model saved at epoch {epoch+1} with loss {best_loss:.4f}")

    if (epoch + 1) % 10 == 0:
        samples = diffusion.ddim_sample(model, 16, IMAGE_SIZE, CHANNELS, ddim_steps=DDIM_STEPS, eta=ETA)
        samples = (samples.clamp(-1, 1) + 1) / 2
        grid = make_grid(samples, nrow=4, normalize=False)
        grid_np = grid.permute(1, 2, 0).cpu().numpy()
        plt.figure(figsize=(8, 8))
        plt.imshow(grid_np)
        plt.axis("off")
        plt.title(f"Epoch {epoch+1} Samples")
        plt.tight_layout()
        plt.savefig(f"/kaggle/working/samples/epoch_{epoch+1:03d}.png", dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Sample grid saved for epoch {epoch+1}")

print("Training complete!")
print(f"Best loss achieved: {best_loss:.4f}")

Starting training...
Epochs: 100 | Batch size: 128 | Device: cuda


Epoch 1/100: 100%|██████████| 390/390 [01:26<00:00,  4.52it/s, loss=0.0532]


Epoch 1/100 | Avg Loss: 0.0955 | LR: 0.000200
New best model saved at epoch 1 with loss 0.0955


Epoch 2/100: 100%|██████████| 390/390 [01:33<00:00,  4.19it/s, loss=0.0476]


Epoch 2/100 | Avg Loss: 0.0432 | LR: 0.000200
New best model saved at epoch 2 with loss 0.0432


...

Epoch 99/100 | Avg Loss: 0.0306 | LR: 0.000000


Epoch 100/100: 100%|██████████| 390/390 [01:32<00:00,  4.20it/s, loss=0.0330]


Epoch 100/100 | Avg Loss: 0.0307 | LR: 0.000000


DDIM Sampling: 100%|██████████| 50/50 [00:00<00:00, 82.02it/s]


Sample grid saved for epoch 100
Training complete!
Best loss achieved: 0.0301


## Phase 5: Evaluation & Results
>This phase produces a complete visual and quantitative evaluation. Four distinct outputs are generated:
>1. Training Loss Curve — plots the MSE loss over all epochs, showing the convergence trajectory. A healthy diffusion model on CIFAR-10 with this architecture should reach roughly 0.03–0.06 MSE after 100 epochs.
2. Final Sample Grid — generates 64 images using the DDIM sampler with 50 steps. These are arranged in an 8×8 grid for visual inspection.
3. Noisy Reconstruction Ladder — takes a real CIFAR-10 image and applies the forward process at various timesteps {0, 200, 400, 600, 800, 999}, then uses the trained model to partially denoise each one back. This directly visualizes what the model has learned: at high noise levels it recovers global structure; at low noise levels it recovers fine detail.
4. FID-Proxy Score — a lightweight proxy for Fréchet Inception Distance. True FID requires 50K samples and a pre-trained Inception network (computationally heavy). Instead, we compute the mean and variance of pixel intensities of generated vs. real images, which gives a rough distributional alignment signal. A proper FID implementation note is included for completeness.

In [5]:
ckpt = torch.load("/kaggle/working/checkpoints/best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Best model loaded from epoch {ckpt['epoch']+1}, loss: {ckpt['loss']:.4f}")


fig, ax = plt.subplots(1, 1, figsize=(12, 5))
ax.plot(range(1, len(train_losses)+1), train_losses, linewidth=2, color="#2563EB")
ax.fill_between(range(1, len(train_losses)+1), train_losses, alpha=0.15, color="#2563EB")
ax.set_xlabel("Epoch", fontsize=13)
ax.set_ylabel("MSE Loss", fontsize=13)
ax.set_title("Diffusion Model Training Loss", fontsize=15, fontweight="bold")
ax.grid(True, alpha=0.3)
ax.set_xlim(1, len(train_losses))
plt.tight_layout()
plt.savefig("/kaggle/working/training_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training loss curve saved.")


print("Generating final 64 sample images via DDIM (50 steps)...")
with torch.no_grad():
    final_samples = diffusion.ddim_sample(
        model, 64, IMAGE_SIZE, CHANNELS, ddim_steps=DDIM_STEPS, eta=ETA
    )

final_samples = (final_samples.clamp(-1, 1) + 1) / 2
grid = make_grid(final_samples, nrow=8, normalize=False, padding=2)
grid_np = grid.permute(1, 2, 0).cpu().numpy()

fig, ax = plt.subplots(1, 1, figsize=(12, 12))
ax.imshow(grid_np)
ax.axis("off")
ax.set_title("DDIM Generated Samples (50 steps) — Best Checkpoint", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/final_samples_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Final sample grid saved.")


real_batch, _ = next(iter(loader))
real_img = real_batch[0].unsqueeze(0).to(DEVICE)

noise_levels = [0, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(2, len(noise_levels), figsize=(16, 6))

for col, t_val in enumerate(noise_levels):
    t_tensor = torch.tensor([t_val], device=DEVICE).long()
    noisy, _ = diffusion.q_sample(real_img, t_tensor)
    noisy_display = (noisy.clamp(-1, 1) + 1) / 2
    axes[0, col].imshow(noisy_display[0].permute(1, 2, 0).cpu().numpy())
    axes[0, col].set_title(f"t={t_val}", fontsize=11)
    axes[0, col].axis("off")

    with torch.no_grad():
        pred_eps = model(noisy, t_tensor)
        ac_t = diffusion.alphas_cumprod[t_val]
        x0_approx = (noisy - torch.sqrt(1 - ac_t) * pred_eps) / torch.sqrt(ac_t)
        x0_approx = (x0_approx.clamp(-1, 1) + 1) / 2

    axes[1, col].imshow(x0_approx[0].permute(1, 2, 0).cpu().numpy())
    axes[1, col].set_title(f"Denoised", fontsize=11)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Noisy Image", fontsize=11, rotation=90)
axes[1, 0].set_ylabel("Model Prediction", fontsize=11, rotation=90)
fig.suptitle("Forward Process Noise Ladder + Model Denoising", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/noise_ladder.png", dpi=150, bbox_inches="tight")
plt.show()
print("Noise ladder visualization saved.")


print("Computing distribution alignment proxy (pixel statistics)...")
with torch.no_grad():
    proxy_samples = diffusion.ddim_sample(
        model, 512, IMAGE_SIZE, CHANNELS, ddim_steps=DDIM_STEPS, eta=ETA
    )
proxy_samples = (proxy_samples.clamp(-1, 1) + 1) / 2

real_data = []
for imgs, _ in loader:
    real_data.append((imgs + 1) / 2)
    if len(real_data) * BATCH_SIZE >= 512:
        break
real_data = torch.cat(real_data, dim=0)[:512]

gen_mean = proxy_samples.mean(dim=[0, 2, 3]).cpu()
gen_std = proxy_samples.std(dim=[0, 2, 3]).cpu()
real_mean = real_data.mean(dim=[0, 2, 3])
real_std = real_data.std(dim=[0, 2, 3])

channels_names = ["Red", "Green", "Blue"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(3)
w = 0.35
axes[0].bar(x_pos - w/2, real_mean.numpy(), w, label="Real", color="#16A34A", alpha=0.85)
axes[0].bar(x_pos + w/2, gen_mean.numpy(), w, label="Generated", color="#DC2626", alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(channels_names)
axes[0].set_ylabel("Mean Pixel Intensity")
axes[0].set_title("Channel Mean Comparison")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(x_pos - w/2, real_std.numpy(), w, label="Real", color="#16A34A", alpha=0.85)
axes[1].bar(x_pos + w/2, gen_std.numpy(), w, label="Generated", color="#DC2626", alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(channels_names)
axes[1].set_ylabel("Std Dev of Pixel Intensity")
axes[1].set_title("Channel Std Dev Comparison")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Real vs Generated Distribution Alignment (Pixel Statistics Proxy)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/distribution_alignment.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n===== FINAL EVALUATION SUMMARY =====")
for i, ch in enumerate(channels_names):
    print(f"{ch} | Real mean: {real_mean[i]:.4f}, Gen mean: {gen_mean[i]:.4f} | "
          f"Real std: {real_std[i]:.4f}, Gen std: {gen_std[i]:.4f}")

print(f"\nBest training loss: {best_loss:.4f}")
print(f"Final training loss: {train_losses[-1]:.4f}")
print(f"Inference: {DDIM_STEPS} DDIM steps (vs 1000 DDPM) = {TOTAL_TIMESTEPS//DDIM_STEPS}x speedup")
print(f"Total samples generated: 64 (grid) + 512 (eval) = 576")
print("=====================================")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 